In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark_training")
    .master("local[2]")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(spark.version)

C:\Users\houar\PycharmProjects\data_engineering_playground\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [3]:
employees = [
    ("Alice", "Data", 42000),
    ("Bob", "Data", 55000),
    ("Charlie", "Software", 48000),
    ("David", "Data", 62000),
    ("Emma", "Software", 51000),
]

employees_df = spark.createDataFrame(
    employees,
    ["name", "department", "salary"]
)

employees_df.show()

+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|  Alice|      Data| 42000|
|    Bob|      Data| 55000|
|Charlie|  Software| 48000|
|  David|      Data| 62000|
|   Emma|  Software| 51000|
+-------+----------+------+



## Exercise 1 — Filtering and selecting data

Using the `employees` DataFrame:

- Keep only employees from the `Data` department.
- Keep only employees with a salary greater than `50,000`.
- Select only the `name` and `salary` columns.
- Sort the result by salary in descending order.

### Expected result

| name  | salary |
|-------|--------|
| David | 62000  |
| Bob   | 55000  |

### Constraints

Use PySpark DataFrame operations only:

- `filter()` or `where()`
- `select()`
- `orderBy()`

Do not use Spark SQL for this exercise.

In [5]:
# Answer 1
df_ex1 = (employees_df
          .filter((employees_df.department == "Data") & (employees_df.salary > 50000))
          .select(["name", "salary"])
          .sort(employees_df.salary.desc()))
df_ex1.show()

+-----+------+
| name|salary|
+-----+------+
|David| 62000|
|  Bob| 55000|
+-----+------+



In general, avoid "collect()" command unless I'm sure the result is small, because it loads the whole list into the driver's memory

## Exercise 2 — Creating a derived column

Using the `employees` DataFrame:

- Create a new column called `salary_level`.
- Set `salary_level` to:
  - `"High"` if `salary >= 55000`
  - `"Medium"` if `salary >= 45000` and `< 55000`
  - `"Low"` if `salary < 45000`
- Keep all original columns.
- Sort the result by salary in descending order.

### Expected result

| name    | department | salary | salary_level |
|---------|------------|--------|--------------|
| David   | Data       | 62000  | High         |
| Bob     | Data       | 55000  | High         |
| Emma    | Software   | 51000  | Medium       |
| Charlie | Software   | 48000  | Medium       |
| Alice   | Data       | 42000  | Low          |

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `withColumn()`
- `when()`
- `otherwise()`
- `orderBy()`

Do not use Spark SQL.

In [7]:
# Answer 1
df_ex2 = (employees_df.withColumn("salary_level",
                                  F.when(employees_df.salary >= 55000, "High")
                                  .otherwise((55000 > employees_df.salary >= 45000), "Medium")
                                  .otherwise(employees_df.salary > 45000, "Low")
                                  )
          .orderBy(employees_df.salary.desc())
          .show()
          )

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

In [ ]:
# Answer 2
df_ex2 = (employees_df.withColumn("salary_level",
                                  F.when(employees_df.salary >= 55000, "High")
                                  .when((55000 > employees_df.salary) & (employees_df.salary >= 45000), "Medium")
                                  .otherwise("Low")
                                  )
          .orderBy(employees_df.salary.desc())
          )

df_ex2.show()

## Exercise 3 — Grouping and aggregating data

Using the `employees` DataFrame:

- Group employees by `department`.
- For each department, calculate:
  - the number of employees
  - the average salary
  - the highest salary
  - the lowest salary
- Rename the resulting columns to:
  - `employee_count`
  - `avg_salary`
  - `max_salary`
  - `min_salary`
- Sort the result by `avg_salary` in descending order.

### Expected result

| department | employee_count | avg_salary | max_salary | min_salary |
|------------|----------------|------------|------------|------------|
| Data       | 3              | 53000.0    | 62000      | 42000      |
| Software   | 2              | 49500.0    | 51000      | 48000      |

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `groupBy()`
- `agg()`
- `count()`
- `avg()`
- `max()`
- `min()`
- `alias()`
- `orderBy()`

Do not use Spark SQL.

In [ ]:
# Answer 1
df_ex3 = (employees_df
          .groupBy("department")
          .withColumn("employee_count", F.count("name"))
          .withColumn("avg_salary", F.aggregate(employees_df.salary.mean()))
          .withColumn("max_salary", F.aggregate(employees_df.salary.max()))
          .withColumn("min_salary", F.aggregate(employees_df.salary.min()))
          .orderBy("avg_salary", ascending=False)
          .show()
          )

In [ ]:
# Answer 2
df_ex3 = (
    employees_df
    .groupBy("department")
    .agg(
        F.count("name").alias("employee_count"),
        F.avg("salary").alias("avg_salary"),
        F.max("salary").alias("max_salary"),
        F.min("salary").alias("min_salary")
    )
    .orderBy("avg_salary", ascending=0)
)

df_ex3.show()

## Exercise 4 — Joining DataFrames

Create a second DataFrame called `departments` using the following data:

| department | manager | location |
|------------|---------|----------|
| Data       | Sarah   | Paris    |
| Software   | Michael | Lyon     |
| HR         | Julie   | Bordeaux |

Then:

- Join the `employees` DataFrame with `departments`.
- Keep only employees whose department exists in both DataFrames.
- Select the following columns:
  - `name`
  - `department`
  - `salary`
  - `manager`
  - `location`
- Sort the result by `department`, then by `salary` in descending order.

### Expected result

| name    | department | salary | manager | location |
|---------|------------|--------|---------|----------|
| David   | Data       | 62000  | Sarah   | Paris    |
| Bob     | Data       | 55000  | Sarah   | Paris    |
| Alice   | Data       | 42000  | Sarah   | Paris    |
| Emma    | Software   | 51000  | Michael | Lyon     |
| Charlie | Software   | 48000  | Michael | Lyon     |

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `spark.createDataFrame()`
- `join()`
- `select()`
- `orderBy()`

Use an `inner` join.

Do not use Spark SQL.

In [ ]:
departments = [
    ("Data", "Sarah", "Paris"),
    ("Software", "Michael", "Lyon"),
    ("HR", "Julie", "Bordeaux")
]

departments_df = spark.createDataFrame(
    departments,
    ["department", "manager", "location"]
)
departments_df.show()

In [ ]:
# Answer 1
df_ex4 = (
    employees_df
    .join(departments_df, on="department", how="inner")
    .select("name", "department", "salary", "manager", "location")
    .orderBy(["department", "salary"], ascending=[1, 0])
)
df_ex4.show()

## Exercise 5 — Join with different column names

Create a new DataFrame called `department_info` using the following data:

| dept_code | manager | budget |
|-----------|---------|--------|
| Data      | Sarah   | 250000 |
| Software  | Michael | 320000 |
| HR        | Julie   | 180000 |

Then:

- Join `employees_df` with `department_info`.
- The join columns have different names:
  - `employees_df.department`
  - `department_info.dept_code`
- Use an `inner` join.
- Keep only:
  - `name`
  - `department`
  - `salary`
  - `manager`
  - `budget`
- Create a new column called `salary_share` equal to:

  `salary / budget`

- Sort by `salary_share` in descending order.

### Expected columns

| name | department | salary | manager | budget | salary_share |
|------|------------|--------|---------|--------|--------------|

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `join()`
- a join condition using two columns
- `select()`
- `withColumn()`
- `orderBy()`

Do not rename `dept_code` before the join.

Do not use Spark SQL.

In [ ]:
department_info = [
    ("Data", "Sarah", "250000"),
    ("Software", "Michael", "320000"),
    ("HR", "Julie", "180000")
]

department_info_df = spark.createDataFrame(
    department_info,
    ["dept_code", "manager", "budget"]
)

department_info_df.show()

In [ ]:
# Answer 1
df_ex5 = (
    employees_df
    .join(department_info_df, on=employees_df.department == department_info_df.dept_code, how="inner")
    .select("name", "department", "salary", "manager", "budget")
    .withColumn("salary_share", F.col("salary") / F.col("budget"))
    .orderBy("salary_share", ascending=0)
)
df_ex5.show()

## Exercise 6 — Handling missing values

Create a new DataFrame called `employees_extended` using the following data:

| name    | department | salary | bonus |
|---------|------------|--------|-------|
| Alice   | Data       | 42000  | 2000  |
| Bob     | Data       | 55000  | null  |
| Charlie | Software   | 48000  | 1500  |
| David   | Data       | 62000  | 3000  |
| Emma    | Software   | 51000  | null  |
| Frank   | null       | 46000  | 1000  |

Then:

- Replace missing values in `bonus` with `0`.
- Remove rows where `department` is missing.
- Create a new column `total_compensation` equal to:

  `salary + bonus`

- Keep:
  - `name`
  - `department`
  - `salary`
  - `bonus`
  - `total_compensation`

- Sort by `total_compensation` in descending order.

### Expected result

| name    | department | salary | bonus | total_compensation |
|---------|------------|--------|-------|--------------------|
| David   | Data       | 62000  | 3000  | 65000              |
| Bob     | Data       | 55000  | 0     | 55000              |
| Emma    | Software   | 51000  | 0     | 51000              |
| Charlie | Software   | 48000  | 1500  | 49500              |
| Alice   | Data       | 42000  | 2000  | 44000              |

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `na.fill()`
- `na.drop()`
- `withColumn()`
- `select()`
- `orderBy()`

Do not use Spark SQL.

In [ ]:
employees_extended = [
    ("Alice", "Data", 42000, 2000),
    ("Bob", "Data", 55000, None),
    ("Charlie", "Software", 48000, 1500),
    ("David", "Data", 62000, 3000),
    ("Emma", "Software", 51000, None),
    ("Frank", None, 46000, 1000)
]

employees_extended_df = spark.createDataFrame(
    employees_extended,
    ["name", "department", "salary", "bonus"]
)

employees_extended_df.show()

In [8]:
# Answer 1
df_ex6 = (
    employees_extended_df
    .fillna(value=0, subset=["bonus"])
    .dropna(how="any", subset=["department"])
    .withColumn("total_compensation", F.col("salary") + F.col("bonus"))
    .select(["name", "department", "salary", "bonus", "total_compensation"])
    .orderBy(F.desc("total_compensation"))
)

df_ex6.show()

NameError: name 'employees_extended_df' is not defined

## Exercise 7 — Data cleaning and aggregation

Create a DataFrame called `sales` using the following data:

| order_id | customer | department | amount | discount |
|----------|----------|------------|--------|----------|
| 1        | Alice    | Data       | 1200   | 100      |
| 2        | Bob      | Data       | 900    | null     |
| 3        | Charlie  | Software   | 1500   | 150      |
| 4        | David    | Data       | 1800   | 200      |
| 5        | Emma     | Software   | 1100   | null     |
| 6        | Alice    | Data       | 700    | 50       |

Then:

- Replace missing values in `discount` with `0`.
- Create a new column `net_amount` equal to:

  `amount - discount`

- Group by `department`.
- For each department, calculate:
  - the number of orders
  - the total `net_amount`
  - the average `net_amount`
  - the maximum `net_amount`

- Rename the output columns to:
  - `order_count`
  - `total_net_amount`
  - `avg_net_amount`
  - `max_net_amount`

- Sort by `total_net_amount` in descending order.

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `fillna()`
- `withColumn()`
- `groupBy()`
- `agg()`
- `count()`
- `sum()`
- `avg()`
- `max()`
- `alias()`
- `orderBy()`

Do not use Spark SQL.

In [9]:
sales = [
    (1, "Alice", "Data", 1200, 100),
    (2, "Bob", "Data", 900, None),
    (3, "Charlie", "Software", 1500, 150),
    (4, "David", "Data", 1800, 200),
    (5, "Emma", "Software", 1100, None),
    (6, "Alice", "Data", 700, 50)
]

sales_df = spark.createDataFrame(
    sales,
    ["order_id", "customer", "department", "amount", "discount"]
)

sales_df.show()

+--------+--------+----------+------+--------+
|order_id|customer|department|amount|discount|
+--------+--------+----------+------+--------+
|       1|   Alice|      Data|  1200|     100|
|       2|     Bob|      Data|   900|    NULL|
|       3| Charlie|  Software|  1500|     150|
|       4|   David|      Data|  1800|     200|
|       5|    Emma|  Software|  1100|    NULL|
|       6|   Alice|      Data|   700|      50|
+--------+--------+----------+------+--------+



In [10]:
# Answer 1
df_ex7 = (
    sales_df
    .fillna(0, subset=["discount"])
    .withColumn("net_amount", F.col("amount") - F.col("discount"))
    .groupBy("department")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("net_amount").alias("total_net_amount"),
        F.mean("net_amount").alias("avg_net_amount"),
        F.max("net_amount").alias("max_net_amount")
    )
    .orderBy(F.desc("total_net_amount"))
)

df_ex7.show()

+----------+-----------+----------------+--------------+--------------+
|department|order_count|total_net_amount|avg_net_amount|max_net_amount|
+----------+-----------+----------------+--------------+--------------+
|      Data|          4|            4250|        1062.5|          1600|
|  Software|          2|            2450|        1225.0|          1350|
+----------+-----------+----------------+--------------+--------------+



## Exercise 8 — End-to-end transformation pipeline

Create the following DataFrames:

### customers

| customer_id | name    | country |
|-------------|---------|---------|
| 1           | Alice   | France  |
| 2           | Bob     | France  |
| 3           | Charlie | Germany |
| 4           | David   | France  |

### orders

| order_id | customer_id | amount | discount |
|----------|-------------|--------|----------|
| 101      | 1           | 1200   | 100      |
| 102      | 2           | 900    | null     |
| 103      | 1           | 700    | 50       |
| 104      | 3           | 1500   | 200      |
| 105      | 4           | 1800   | null     |
| 106      | 3           | 500    | 0        |

Then:

- Replace missing `discount` values with `0`.
- Create a `net_amount` column equal to `amount - discount`.
- Join `orders` with `customers` using `customer_id`.
- Keep only customers from `France`.
- Group by customer.
- Calculate:
  - number of orders
  - total net amount
  - average net amount
- Rename the aggregated columns.
- Sort by total net amount in descending order.

### Expected columns

| customer_id | name | order_count | total_net_amount | avg_net_amount |
|-------------|------|-------------|------------------|----------------|

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `fillna()`
- `withColumn()`
- `join()`
- `filter()`
- `groupBy()`
- `agg()`
- `alias()`
- `orderBy()`

Do not use Spark SQL.

In [11]:
customers = [
    (1, "Alice", "France"),
    (2, "Bob", "France"),
    (3, "Charlie", "Germany"),
    (4, "David", "France")
]

orders = [
    (101, 1, 1200, 100),
    (102, 2, 900, None),
    (103, 1, 700, 50),
    (104, 3, 1500, 200),
    (105, 5, 1800, None),
    (106, 3, 500, 0)
]

customers_df = spark.createDataFrame(
    customers,
    ["customer_id", "name", "country"]
)
customers_df.show()

orders_df = spark.createDataFrame(
    orders,
    ["order_id", "customer_id", "amount", "discount"]
)
orders_df.show()

+-----------+-------+-------+
|customer_id|   name|country|
+-----------+-------+-------+
|          1|  Alice| France|
|          2|    Bob| France|
|          3|Charlie|Germany|
|          4|  David| France|
+-----------+-------+-------+

+--------+-----------+------+--------+
|order_id|customer_id|amount|discount|
+--------+-----------+------+--------+
|     101|          1|  1200|     100|
|     102|          2|   900|    NULL|
|     103|          1|   700|      50|
|     104|          3|  1500|     200|
|     105|          5|  1800|    NULL|
|     106|          3|   500|       0|
+--------+-----------+------+--------+



In [12]:
orders_df = (
    orders_df
    .fillna(0, subset=["discount"])
    .withColumn("net_amount", F.col("amount") - F.col("discount"))
)

df_ex8 = (
    orders_df
    .join(customers_df, how="inner", on="customer_id")
    .filter(F.col("country") == "France")
    .groupBy("customer_id", "name")
    .agg(
        F.count("order_id").alias("number_of_orders"),
        F.sum("net_amount").alias("total_net_amount"),
        F.mean("net_amount").alias("avg_net_amount")
    )
    .orderBy(F.desc("total_net_amount"))
)
df_ex8.show()

+-----------+-----+----------------+----------------+--------------+
|customer_id| name|number_of_orders|total_net_amount|avg_net_amount|
+-----------+-----+----------------+----------------+--------------+
|          1|Alice|               2|            1750|         875.0|
|          2|  Bob|               1|             900|         900.0|
+-----------+-----+----------------+----------------+--------------+



## Exercise 9 — Removing duplicates and combining DataFrames

Create two DataFrames: `orders_january` and `orders_february`.

### orders_january

| order_id | customer | amount |
|----------|----------|--------|
| 101      | Alice    | 1200   |
| 102      | Bob      | 900    |
| 103      | Charlie  | 1500   |
| 103      | Charlie  | 1500   |
| 104      | David    | 1800   |

### orders_february

| order_id | customer | amount |
|----------|----------|--------|
| 105      | Emma     | 1100   |
| 106      | Alice    | 700    |
| 107      | Bob      | 1300   |
| 107      | Bob      | 1300   |

### Part 1 — Inspect duplicates

Using `orders_january`:

- Display only the unique rows.
- Count the number of rows before removing duplicates.
- Count the number of rows after removing duplicates.

### Part 2 — Remove duplicates

Create a DataFrame called `january_clean`:

- Remove duplicate rows from `orders_january`.
- Keep one copy of each duplicated row.

### Part 3 — Combine both months

Create a DataFrame called `all_orders`:

- Remove duplicates from `orders_february`.
- Combine `january_clean` with the cleaned February DataFrame.
- Sort the final DataFrame by `order_id`.

### Part 4 — Customer-level deduplication

From `all_orders`, create another DataFrame called `unique_customers`:

- Keep only one row per customer.
- Do not worry which order is retained for customers with several orders.

### Concepts to practice

Try to use:

- `distinct()`
- `dropDuplicates()`
- `unionByName()`
- `count()`
- `orderBy()`

Do not use Spark SQL.

In [13]:
orders_df_columns = ["order_id", "customer", "amount"]

orders_january = [
    (101, "Alice", 1200),
    (102, "Bob", 900),
    (103, "Charlie", 1500),
    (103, "Charlie", 1500),
    (104, "David", 1800)
]

orders_february = [
    (105, "Emma", 1100),
    (106, "Alice", 700),
    (107, "Bob", 1300),
    (107, "Bob", 1300)
]

orders_january_df = spark.createDataFrame(
    orders_january,
    orders_df_columns
)

orders_february_df = spark.createDataFrame(
    orders_february,
    orders_df_columns
)

orders_january_df.show()
orders_february_df.show()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|     101|   Alice|  1200|
|     102|     Bob|   900|
|     103| Charlie|  1500|
|     103| Charlie|  1500|
|     104|   David|  1800|
+--------+--------+------+

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|     105|    Emma|  1100|
|     106|   Alice|   700|
|     107|     Bob|  1300|
|     107|     Bob|  1300|
+--------+--------+------+



In [14]:
# Part 1&2
# Answer 1
nb_rows_b_dedup = orders_january_df.count()
orders_january_dedup_df = (
    orders_january_df
    .distinct()
)
orders_january_dedup_df.show()
nb_rows_a_dedup = orders_january_dedup_df.count()

print(f"Rows before/after deduplication: {nb_rows_b_dedup} / {nb_rows_a_dedup}")

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|     101|   Alice|  1200|
|     102|     Bob|   900|
|     103| Charlie|  1500|
|     104|   David|  1800|
+--------+--------+------+

Rows before/after deduplication: 5 / 4


In [15]:
# Part 3
# Answer 1
orders_february_dedup_df = (
    orders_february_df
    .distinct()
)

all_orders = (
    orders_february_dedup_df
    .unionByName(orders_january_dedup_df)
    .orderBy(F.asc("order_id"))
)
all_orders.show()


+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|     101|   Alice|  1200|
|     102|     Bob|   900|
|     103| Charlie|  1500|
|     104|   David|  1800|
|     105|    Emma|  1100|
|     106|   Alice|   700|
|     107|     Bob|  1300|
+--------+--------+------+



In [16]:
# Part 4
# Answer 1
unique_customers = (
    all_orders
    .dropDuplicates(["customer"])
)
unique_customers.show()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|     101|   Alice|  1200|
|     104|   David|  1800|
|     105|    Emma|  1100|
|     102|     Bob|   900|
|     103| Charlie|  1500|
+--------+--------+------+



## Exercise 10 — Working with dates and timestamps

Create a DataFrame called `transactions` using the following data:

| transaction_id | customer | transaction_date | amount |
|----------------|----------|------------------|--------|
| 1              | Alice    | 2026-07-01       | 1200   |
| 2              | Bob      | 2026-07-03       | 900    |
| 3              | Charlie  | 2026-07-10       | 1500   |
| 4              | Alice    | 2026-07-15       | 700    |
| 5              | Bob      | 2026-07-20       | 1300   |

Then:

- Convert `transaction_date` from string to a real date column.
- Create:
  - `year`
  - `month`
  - `day`
- Create a new column `days_since_transaction` representing the number of days between `2026-07-31` and `transaction_date`.
- Keep only transactions from July 2026.
- Sort by `transaction_date` in descending order.

### Expected columns

| transaction_id | customer | transaction_date | amount | year | month | day | days_since_transaction |
|----------------|----------|------------------|--------|------|-------|-----|------------------------|

### Constraints

Use PySpark DataFrame operations only.

Try to use:

- `to_date()`
- `year()`
- `month()`
- `dayofmonth()`
- `datediff()`
- `lit()`
- `filter()`
- `orderBy()`

Do not use Spark SQL.


Note: the dates were slightly modified compared to the exercice instructions, to get a clearer view of the filters' effect

In [17]:
transactions = [
    (1, "Alice", "2026-07-01", 1200),
    (2, "Bob", "2026-06-03", 900),
    (3, "Charlie", "2026-07-10", 1500),
    (4, "Alice", "2026-07-15", 700),
    (5, "Bob", "2026-08-20", 1300),
]

transactions_df = spark.createDataFrame(
    transactions,
    ["transaction_id", "customer", "transaction_date", "amount"]
)

transactions_df.show()

+--------------+--------+----------------+------+
|transaction_id|customer|transaction_date|amount|
+--------------+--------+----------------+------+
|             1|   Alice|      2026-07-01|  1200|
|             2|     Bob|      2026-06-03|   900|
|             3| Charlie|      2026-07-10|  1500|
|             4|   Alice|      2026-07-15|   700|
|             5|     Bob|      2026-08-20|  1300|
+--------------+--------+----------------+------+



In [18]:
# Answer 1
transactions_df = (
    transactions_df
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
    .withColumn("year", F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
    .withColumn("day", F.dayofmonth(F.col("transaction_date")))
    .withColumn(
        "days_since_transaction",
        F.datediff(
            F.lit("2026-07-31"),
            F.col("transaction_date")
        )
    )
    .filter(F.col("transaction_date") > "2026-06-30")
    .orderBy(F.desc("transaction_date"))
)

transactions_df.show()

+--------------+--------+----------------+------+----+-----+---+----------------------+
|transaction_id|customer|transaction_date|amount|year|month|day|days_since_transaction|
+--------------+--------+----------------+------+----+-----+---+----------------------+
|             5|     Bob|      2026-08-20|  1300|2026|    8| 20|                   -20|
|             4|   Alice|      2026-07-15|   700|2026|    7| 15|                    16|
|             3| Charlie|      2026-07-10|  1500|2026|    7| 10|                    21|
|             1|   Alice|      2026-07-01|  1200|2026|    7|  1|                    30|
+--------------+--------+----------------+------+----+-----+---+----------------------+



In [19]:
# Answer 2
transactions_df = (
    transactions_df
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
    .withColumn("year", F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
    .withColumn("day", F.dayofmonth(F.col("transaction_date")))
    .withColumn(
        "days_since_transaction",
        F.datediff(
            F.lit("2026-07-31"),
            F.col("transaction_date")
        )
    )
    .filter(
        (F.col("transaction_date") >= "2026-07-01") &
        (F.col("transaction_date") < "2026-08-01")
    )
    .orderBy(F.desc("transaction_date"))
)

transactions_df.show()

+--------------+--------+----------------+------+----+-----+---+----------------------+
|transaction_id|customer|transaction_date|amount|year|month|day|days_since_transaction|
+--------------+--------+----------------+------+----+-----+---+----------------------+
|             4|   Alice|      2026-07-15|   700|2026|    7| 15|                    16|
|             3| Charlie|      2026-07-10|  1500|2026|    7| 10|                    21|
|             1|   Alice|      2026-07-01|  1200|2026|    7|  1|                    30|
+--------------+--------+----------------+------+----+-----+---+----------------------+



## Exercise 11 — Ranking rows within each group

Create the following DataFrame `sales_rank`:

| order_id | department | employee | amount |
|----------|------------|----------|--------|
| 1        | Data       | Alice    | 1200   |
| 2        | Data       | Bob      | 1800   |
| 3        | Data       | David    | 900    |
| 4        | Software   | Emma     | 1500   |
| 5        | Software   | Charlie  | 1700   |
| 6        | Software   | Michael  | 1100   |

Then:

- Rank employees within each `department`.
- The highest `amount` should have rank `1`.
- Create a new column called `rank`.
- Keep all original columns.
- Sort the final result by `department` and `rank`.

### Expected result

| department | employee | amount | rank |
|------------|----------|--------|------|
| Data       | Bob      | 1800   | 1    |
| Data       | Alice    | 1200   | 2    |
| Data       | David    | 900    | 3    |
| Software   | Charlie  | 1700   | 1    |
| Software   | Emma     | 1500   | 2    |
| Software   | Michael  | 1100   | 3    |

### Concepts to use

- `Window`
- `partitionBy()`
- `orderBy()`
- `row_number()`
- `withColumn()`

In [21]:
sales_rank_data = [
    (1, "Data", "Alice", 1200),
    (2, "Data", "Bob", 1800),
    (3, "Data", "David", 900),
    (4, "Software", "Emma", 1500),
    (5, "Software", "Charlie", 1700),
    (6, "Software", "Michael", 1100),
]

sales_rank_df = spark.createDataFrame(
    sales_rank_data,
    ["order_id", "department", "employee", "amount"]
)

sales_rank_df.show()

+--------+----------+--------+------+
|order_id|department|employee|amount|
+--------+----------+--------+------+
|       1|      Data|   Alice|  1200|
|       2|      Data|     Bob|  1800|
|       3|      Data|   David|   900|
|       4|  Software|    Emma|  1500|
|       5|  Software| Charlie|  1700|
|       6|  Software| Michael|  1100|
+--------+----------+--------+------+



In [23]:
# Answer 1
window = Window.partitionBy("department").orderBy(F.desc("amount"))

df_ex11 = (
    sales_rank_df
    .withColumn("row_number", F.row_number().over(window))
    .withColumn("rank", F.rank().over(window))
    .orderBy([F.col("department"), F.col("rank")])
)

df_ex11.show()

+--------+----------+--------+------+----------+----+
|order_id|department|employee|amount|row_number|rank|
+--------+----------+--------+------+----------+----+
|       2|      Data|     Bob|  1800|         1|   1|
|       1|      Data|   Alice|  1200|         2|   2|
|       3|      Data|   David|   900|         3|   3|
|       5|  Software| Charlie|  1700|         1|   1|
|       4|  Software|    Emma|  1500|         2|   2|
|       6|  Software| Michael|  1100|         3|   3|
+--------+----------+--------+------+----------+----+



In [24]:
window = Window.partitionBy("department").orderBy(F.desc("amount"))

df_ex11 = (
    sales_rank_df
    .withColumn("row_number", F.row_number().over(window))
    .withColumn("rank", F.rank().over(window))
    .orderBy([F.col("department"), F.col("rank")])
)

df_ex11.show()

+--------+----------+--------+------+----------+----+
|order_id|department|employee|amount|row_number|rank|
+--------+----------+--------+------+----------+----+
|       2|      Data|     Bob|  1800|         1|   1|
|       1|      Data|   Alice|  1200|         2|   2|
|       3|      Data|   David|   900|         3|   3|
|       5|  Software| Charlie|  1700|         1|   1|
|       4|  Software|    Emma|  1500|         2|   2|
|       6|  Software| Michael|  1100|         3|   3|
+--------+----------+--------+------+----------+----+



## Exercise 12 — Comparing `row_number`, `rank` and `dense_rank`

Create the following DataFrame `sales_ties`:

| order_id | department | employee | amount |
|----------|------------|----------|--------|
| 1        | Data       | Alice    | 1800   |
| 2        | Data       | Bob      | 1800   |
| 3        | Data       | David    | 1200   |
| 4        | Data       | Sarah    | 900    |
| 5        | Software   | Emma     | 1700   |
| 6        | Software   | Charlie  | 1700   |
| 7        | Software   | Michael  | 1100   |

Then:

- Partition by `department`.
- Sort by `amount` in descending order.
- Create three new columns:
  - `row_number`
  - `rank`
  - `dense_rank`
- Sort the final result by `department`, then by `amount` descending.

### Goal

Observe how the three functions behave when two employees have the same `amount`.

### Expected behavior for Data

| employee | amount | row_number | rank | dense_rank |
|----------|--------|------------|------|------------|
| Alice    | 1800   | 1          | 1    | 1          |
| Bob      | 1800   | 2          | 1    | 1          |
| David    | 1200   | 3          | 3    | 2          |
| Sarah    | 900    | 4          | 4    | 3          |

### Concepts to use

- `Window.partitionBy()`
- `orderBy()`
- `row_number()`
- `rank()`
- `dense_rank()`
- `over()`

In [25]:
sales_ties_data = [
    (1, "Data", "Alice", 1800),
    (2, "Data", "Bob", 1800),
    (3, "Data", "David", 1200),
    (4, "Data", "Sarah", 900),
    (5, "Software", "Emma", 1700),
    (6, "Software", "Charlie", 1700),
    (7, "Software", "Michael", 1100),
]

sales_ties_df = spark.createDataFrame(
    sales_ties_data,
    ["order_id", "department", "employee", "amount"]
)

In [28]:
window_ties = Window.partitionBy("department").orderBy(F.desc("amount"))

df_ex12 = (
    sales_ties_df
    .withColumn("row_number", F.row_number().over(window_ties))
    .withColumn("rank", F.rank().over(window_ties))
    .withColumn("dense_rank", F.dense_rank().over(window_ties))
    .orderBy(["department", "amount"], ascending=[True, False])
)
df_ex12.show()

+--------+----------+--------+------+----------+----+----------+
|order_id|department|employee|amount|row_number|rank|dense_rank|
+--------+----------+--------+------+----------+----+----------+
|       1|      Data|   Alice|  1800|         1|   1|         1|
|       2|      Data|     Bob|  1800|         2|   1|         1|
|       3|      Data|   David|  1200|         3|   3|         2|
|       4|      Data|   Sarah|   900|         4|   4|         3|
|       5|  Software|    Emma|  1700|         1|   1|         1|
|       6|  Software| Charlie|  1700|         2|   1|         1|
|       7|  Software| Michael|  1100|         3|   3|         2|
+--------+----------+--------+------+----------+----+----------+



## Exercise 13 — Comparing each transaction with the previous one

Create a DataFrame called `customer_transactions`:

| transaction_id | customer | transaction_date | amount |
|----------------|----------|------------------|--------|
| 1              | Alice    | 2026-07-01       | 1000   |
| 2              | Alice    | 2026-07-08       | 1300   |
| 3              | Alice    | 2026-07-20       | 900    |
| 4              | Bob      | 2026-07-03       | 700    |
| 5              | Bob      | 2026-07-15       | 1200   |
| 6              | Bob      | 2026-07-25       | 1500   |

Then:

- Convert `transaction_date` to a date.
- Partition the data by `customer`.
- Order each customer's transactions by `transaction_date`.
- Create a column `previous_amount` containing the amount of the previous transaction.
- Create a column `amount_difference` equal to:

  `amount - previous_amount`

- Sort the final result by `customer` and `transaction_date`.

### Concepts to use

- `Window.partitionBy()`
- `orderBy()`
- `lag()`
- `withColumn()`
- `to_date()`

Do not use Spark SQL.

In [2]:
customer_transactions_data = [
    (1, "Alice", "2026-07-01", 1000),
    (2, "Alice", "2026-07-08", 1300),
    (3, "Alice", "2026-07-20", 900),
    (4, "Bob", "2026-07-03", 700),
    (5, "Bob", "2026-07-15", 1200),
    (6, "Bob", "2026-07-25", 1500),
]

customer_transactions_df = spark.createDataFrame(
    customer_transactions_data,
    ["transaction_id", "customer", "transaction_date", "amount"]
)

In [9]:
# Answer 1
window_transactions = Window.partitionBy("customer").orderBy("transaction_date")

df_ex13 = (
    customer_transactions_df
    .withColumn("row_number", F.row_number().over(window_transactions))
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("previous_amount", F.lag("amount", offset=1).over(window_transactions))
    .withColumn("amount_difference", F.col("amount") - F.col("previous_amount"))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_ex13.show()

+--------------+--------+----------------+------+----------+---------------+-----------------+
|transaction_id|customer|transaction_date|amount|row_number|previous_amount|amount_difference|
+--------------+--------+----------------+------+----------+---------------+-----------------+
|             1|   Alice|      2026-07-01|  1000|         1|           NULL|             NULL|
|             2|   Alice|      2026-07-08|  1300|         2|           1000|              300|
|             3|   Alice|      2026-07-20|   900|         3|           1300|             -400|
|             4|     Bob|      2026-07-03|   700|         1|           NULL|             NULL|
|             5|     Bob|      2026-07-15|  1200|         2|            700|              500|
|             6|     Bob|      2026-07-25|  1500|         3|           1200|              300|
+--------------+--------+----------------+------+----------+---------------+-----------------+



In [10]:
# Answer 2
window_transactions = Window.partitionBy("customer").orderBy("transaction_date")

df_ex13 = (
    customer_transactions_df
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("previous_amount", F.lag("amount", offset=1).over(window_transactions))
    .withColumn("amount_difference", F.col("amount") - F.col("previous_amount"))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_ex13.show()

+--------------+--------+----------------+------+---------------+-----------------+
|transaction_id|customer|transaction_date|amount|previous_amount|amount_difference|
+--------------+--------+----------------+------+---------------+-----------------+
|             1|   Alice|      2026-07-01|  1000|           NULL|             NULL|
|             2|   Alice|      2026-07-08|  1300|           1000|              300|
|             3|   Alice|      2026-07-20|   900|           1300|             -400|
|             4|     Bob|      2026-07-03|   700|           NULL|             NULL|
|             5|     Bob|      2026-07-15|  1200|            700|              500|
|             6|     Bob|      2026-07-25|  1500|           1200|              300|
+--------------+--------+----------------+------+---------------+-----------------+



## Exercise 14 — Cumulative transaction amount

- Convert `transaction_date` to a date.
- Partition the data by `customer`.
- Order each customer's transactions by `transaction_date`.
- Add a column called `cumulative_amount`.
- This column must contain the sum of all transaction amounts from the
  customer's first transaction up to the current transaction.
- Sort the result by `customer` and `transaction_date`.

### Expected result for Alice

| transaction_date | amount | cumulative_amount |
|------------------|--------|-------------------|
| 2026-07-01       | 1000   | 1000              |
| 2026-07-08       | 1300   | 2300              |
| 2026-07-20       | 900    | 3200              |

### Concepts to use

- `Window.partitionBy()`
- `orderBy()`
- `rowsBetween()`
- `Window.unboundedPreceding`
- `Window.currentRow`
- `sum().over()`

In [11]:
# Answer 1
window_ex14 = Window.partitionBy("customer").orderBy("transaction_date")

df_ex14 = (
    customer_transactions_df
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("cumulative_amount", F.sum("amount").over(window_ex14))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_ex14.show()


+--------------+--------+----------------+------+-----------------+
|transaction_id|customer|transaction_date|amount|cumulative_amount|
+--------------+--------+----------------+------+-----------------+
|             1|   Alice|      2026-07-01|  1000|             1000|
|             2|   Alice|      2026-07-08|  1300|             2300|
|             3|   Alice|      2026-07-20|   900|             3200|
|             4|     Bob|      2026-07-03|   700|              700|
|             5|     Bob|      2026-07-15|  1200|             1900|
|             6|     Bob|      2026-07-25|  1500|             3400|
+--------------+--------+----------------+------+-----------------+



In [12]:
# Answer 2 (correction - not from me)
window_ex14 = (
    Window.partitionBy("customer")
    .orderBy("transaction_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_ex14 = (
    customer_transactions_df
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("cumulative_amount", F.sum("amount").over(window_ex14))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_ex14.show()

+--------------+--------+----------------+------+-----------------+
|transaction_id|customer|transaction_date|amount|cumulative_amount|
+--------------+--------+----------------+------+-----------------+
|             1|   Alice|      2026-07-01|  1000|             1000|
|             2|   Alice|      2026-07-08|  1300|             2300|
|             3|   Alice|      2026-07-20|   900|             3200|
|             4|     Bob|      2026-07-03|   700|              700|
|             5|     Bob|      2026-07-15|  1200|             1900|
|             6|     Bob|      2026-07-25|  1500|             3400|
+--------------+--------+----------------+------+-----------------+



Adding `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` tells spark to make the window from the first row to the current one.
When not using this, if two rows have the same date (or same value used as partition), then they get the same result.
Below a full exemple

In [17]:
customer_transactions_data_test = [
    (1, "Alice", "2026-07-01", 1000),
    (2, "Alice", "2026-07-08", 1300),
    (3, "Alice", "2026-07-08", 900),
    (4, "Bob", "2026-07-15", 700),
    (5, "Bob", "2026-07-15", 1200),
    (6, "Bob", "2026-07-25", 1500),
]

customer_transactions_data_test_df = spark.createDataFrame(
    customer_transactions_data_test,
    ["transaction_id", "customer", "transaction_date", "amount"]
)

In [18]:
# Without .rows_between
window_test_ex14 = Window.partitionBy("customer").orderBy("transaction_date")

df_test_ex14 = (
    customer_transactions_data_test_df
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("cumulative_amount", F.sum("amount").over(window_test_ex14))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_test_ex14.show()

+--------------+--------+----------------+------+-----------------+
|transaction_id|customer|transaction_date|amount|cumulative_amount|
+--------------+--------+----------------+------+-----------------+
|             1|   Alice|      2026-07-01|  1000|             1000|
|             2|   Alice|      2026-07-08|  1300|             3200|
|             3|   Alice|      2026-07-08|   900|             3200|
|             4|     Bob|      2026-07-15|   700|             1900|
|             5|     Bob|      2026-07-15|  1200|             1900|
|             6|     Bob|      2026-07-25|  1500|             3400|
+--------------+--------+----------------+------+-----------------+



In [21]:
# With .rows_between
window_test2_ex14 = (
    Window.partitionBy("customer")
    .orderBy("transaction_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_test2_ex14 = (
    customer_transactions_data_test_df
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("cumulative_amount", F.sum("amount").over(window_test2_ex14))
    .orderBy(["customer", "transaction_date"], ascending=[True, True])
)
df_test2_ex14.show()

+--------------+--------+----------------+------+-----------------+
|transaction_id|customer|transaction_date|amount|cumulative_amount|
+--------------+--------+----------------+------+-----------------+
|             1|   Alice|      2026-07-01|  1000|             1000|
|             2|   Alice|      2026-07-08|  1300|             2300|
|             3|   Alice|      2026-07-08|   900|             3200|
|             4|     Bob|      2026-07-15|   700|              700|
|             5|     Bob|      2026-07-15|  1200|             1900|
|             6|     Bob|      2026-07-25|  1500|             3400|
+--------------+--------+----------------+------+-----------------+



## Exercise 15 — Window Functions with Spark SQL

Using the existing `customer_transactions_df` DataFrame, create a temporary SQL view named `customer_transactions`.

Then write a Spark SQL query that:

- Converts `transaction_date` to a date.
- Partitions the data by `customer`.
- Orders each customer's transactions by `transaction_date`.
- Creates a column named `previous_amount` containing the amount of the previous transaction.
- Creates a column named `amount_difference` equal to:

  `amount - previous_amount`

- Sorts the final result by `customer` and `transaction_date`.

### Expected result for Alice

| customer | transaction_date | amount | previous_amount | amount_difference |
|----------|------------------|--------|-----------------|-------------------|
| Alice    | 2026-07-01       | 1000   | null            | null              |
| Alice    | 2026-07-08       | 1300   | 1000            | 300               |
| Alice    | 2026-07-20       | 900    | 1300            | -400              |

### Concepts to use

- `createOrReplaceTempView()`
- `spark.sql()`
- `CAST(... AS DATE)`
- `LAG()`
- `OVER()`
- `PARTITION BY`
- `ORDER BY`

### Starter code

```python
customer_transactions_df.createOrReplaceTempView("customer_transactions")

df_ex15 = spark.sql("""
    SELECT
        ...
    FROM customer_transactions
    ORDER BY ...
""")

df_ex15.show()

In [18]:
# Answer 1
customer_transactions_df.createOrReplaceTempView("customer_transactions")

df_ex15 = spark.sql(
    """
    WITH transactions_with_previous AS (
        SELECT
            transaction_id,
            customer,
            CAST(transaction_date AS DATE) AS transaction_date,
            amount,
            LAG(amount, 1) OVER(
                PARTITION BY customer
                ORDER BY CAST(transaction_date AS DATE)
            ) AS previous_amount
        FROM customer_transactions
    )

    SELECT *, amount - previous_amount AS amount_difference
    FROM transactions_with_previous
    ORDER BY customer, transaction_date
    """
)

df_ex15.show()

+--------------+--------+----------------+------+---------------+-----------------+
|transaction_id|customer|transaction_date|amount|previous_amount|amount_difference|
+--------------+--------+----------------+------+---------------+-----------------+
|             1|   Alice|      2026-07-01|  1000|           NULL|             NULL|
|             2|   Alice|      2026-07-08|  1300|           1000|              300|
|             3|   Alice|      2026-07-20|   900|           1300|             -400|
|             4|     Bob|      2026-07-03|   700|           NULL|             NULL|
|             5|     Bob|      2026-07-15|  1200|            700|              500|
|             6|     Bob|      2026-07-25|  1500|           1200|              300|
+--------------+--------+----------------+------+---------------+-----------------+

